# ToolCallingAgent com smolagents: Agentes com Ferramentas

> **Artigo:** [ToolCallingAgent com smolagents: Agentes com Ferramentas](https://iablog.github.io)  
> **Autor:** Sarah P. Lima  
> **Data:** 23/05/2026

Neste notebook vamos construir um assistente de agendamento médico completo usando o `ToolCallingAgent` da biblioteca smolagents. O agente será capaz de:

- Listar especialidades e médicos disponíveis
- Verificar horários livres de um médico
- Agendar consultas para pacientes
- Mostrar um resumo dos agendamentos feitos

## 1. Instalação

In [ ]:
!pip install -q "smolagents[litellm]"

## 2. Configuração do Modelo

Aqui usamos o Gemini 2.5 Flash via LiteLLM. Você precisa de uma `GOOGLE_API_KEY` salva nos Secrets do Colab (`🔑` no menu lateral).

In [ ]:
from smolagents import ToolCallingAgent, LiteLLMModel
from google.colab import userdata

model = LiteLLMModel(
    model_id="gemini/gemini-2.5-flash",
    api_key=userdata.get("GOOGLE_API_KEY"),
    timeout=60
)

## 3. Dados da Clínica

`MEDICOS` e `AGENDA` simulam um banco de dados. `CONSULTAS_AGENDADAS` acumula os agendamentos feitos durante a sessão.

In [ ]:
MEDICOS = {
    "Dr. Carlos Mendes": {"especialidade": "Cardiologia", "crm": "PE-12345"},
    "Dra. Ana Lima":     {"especialidade": "Dermatologia", "crm": "PE-67890"},
    "Dr. Pedro Souza":   {"especialidade": "Ortopedia",   "crm": "PE-11223"},
    "Dra. Julia Costa":  {"especialidade": "Pediatria",   "crm": "PE-44556"},
}

AGENDA = {
    "Dr. Carlos Mendes": ["08:00", "09:00", "14:00", "15:00"],
    "Dra. Ana Lima":     ["10:00", "11:00", "16:00"],
    "Dr. Pedro Souza":   ["08:00", "13:00", "14:00"],
    "Dra. Julia Costa":  ["09:00", "10:00", "11:00", "15:00"],
}

CONSULTAS_AGENDADAS = []

## 4. Definindo as Ferramentas com `@tool`

O decorator `@tool` transforma uma função Python comum em uma ferramenta que o agente pode chamar. Dois elementos são essenciais:

- **A docstring**: o agente lê isso para decidir *quando* e *como* usar cada ferramenta
- **As type annotations**: o agente usa isso para passar os argumentos no tipo correto

In [ ]:
from smolagents import tool

@tool
def listar_especialidades() -> str:
    """
    Lista todas as especialidades médicas disponíveis na clínica
    e os respectivos médicos de cada especialidade.
    Use esta ferramenta quando o paciente perguntar quais médicos
    ou especialidades estão disponíveis.
    """
    resultado = "Especialidades disponíveis:\n"
    for medico, dados in MEDICOS.items():
        resultado += f"- {dados['especialidade']}: {medico}\n"
    return resultado


@tool
def verificar_horarios(nome_medico: str) -> str:
    """
    Retorna os horários disponíveis de um médico específico.
    Use esta ferramenta antes de agendar, para verificar
    se há horários livres.

    Args:
        nome_medico: Nome completo do médico conforme listado
                     em listar_especialidades().
    """
    if nome_medico not in AGENDA:
        return f"Médico '{nome_medico}' não encontrado. Use listar_especialidades() para ver os nomes corretos."
    horarios = AGENDA[nome_medico]
    if not horarios:
        return f"{nome_medico} não possui horários disponíveis no momento."
    return f"Horários disponíveis de {nome_medico}: {', '.join(horarios)}"


@tool
def agendar_consulta(nome_paciente: str, nome_medico: str, horario: str) -> str:
    """
    Agenda uma consulta para o paciente com o médico no horário indicado.
    Só chame esta ferramenta após confirmar com verificar_horarios()
    que o horário está disponível.

    Args:
        nome_paciente: Nome completo do paciente.
        nome_medico: Nome completo do médico.
        horario: Horário no formato HH:MM (ex: '09:00').
    """
    if nome_medico not in AGENDA:
        return f"Médico '{nome_medico}' não encontrado."
    if horario not in AGENDA[nome_medico]:
        return f"Horário {horario} não está disponível para {nome_medico}."

    AGENDA[nome_medico].remove(horario)
    especialidade = MEDICOS[nome_medico]["especialidade"]
    CONSULTAS_AGENDADAS.append({
        "paciente": nome_paciente,
        "medico": nome_medico,
        "especialidade": especialidade,
        "horario": horario
    })
    return (f"Consulta agendada com sucesso!\n"
            f"Paciente: {nome_paciente}\n"
            f"Médico: {nome_medico} ({especialidade})\n"
            f"Horário: {horario}")


@tool
def ver_agendamentos() -> str:
    """
    Mostra todas as consultas agendadas na sessão atual.
    Use no final do atendimento ou quando o paciente pedir
    um resumo do que foi marcado.
    """
    if not CONSULTAS_AGENDADAS:
        return "Nenhuma consulta agendada ainda."
    resultado = "Consultas agendadas:\n"
    for c in CONSULTAS_AGENDADAS:
        resultado += (f"- {c['paciente']} com {c['medico']} "
                      f"({c['especialidade']}) às {c['horario']}\n")
    return resultado

## 5. Criando o Agente

In [ ]:
agente_clinica = ToolCallingAgent(
    tools=[listar_especialidades, verificar_horarios,
           agendar_consulta, ver_agendamentos],
    model=model
)

### Inspecionando o system prompt

O smolagents monta automaticamente um system prompt a partir das ferramentas. Cada ferramenta vira um bloco com nome, descrição e parâmetros — é por isso que a docstring importa.

In [ ]:
print(agente_clinica.system_prompt)

## 6. Conversa em Múltiplos Turnos

O `ToolCallingAgent` suporta conversas com memória usando `reset=False`. A partir da segunda mensagem, o agente lembra o que foi dito e feito anteriormente.

### Turno 1 — O que está disponível?

In [ ]:
from IPython.display import Markdown

resposta1 = agente_clinica.run(
    "Olá! Preciso marcar uma consulta. "
    "Quais especialidades vocês têm disponíveis?"
)
Markdown(resposta1)

### Turno 2 — Verificando horários

In [ ]:
resposta2 = agente_clinica.run(
    "Quero Cardiologia. Quais horários o Dr. Carlos Mendes tem disponível?",
    reset=False
)
Markdown(resposta2)

### Turno 3 — Agendando

In [ ]:
resposta3 = agente_clinica.run(
    "Pode marcar para Maria Silva às 14h.",
    reset=False
)
Markdown(resposta3)

### Turno 4 — Agendando mais uma e pedindo resumo

Neste turno o agente recebe uma instrução que exige três ferramentas em sequência: verificar horários → agendar → mostrar resumo.

In [ ]:
resposta4 = agente_clinica.run(
    "Sim! Quero também marcar Pediatria para o João Silva. "
    "Verifique os horários disponíveis e marque às 10h. "
    "Depois me mostre o resumo de tudo que foi agendado.",
    reset=False
)
Markdown(resposta4)

## 7. Verificando o Estado Real

Confirmando que as ferramentas realmente modificaram o sistema — os horários foram removidos da `AGENDA` e os registros foram adicionados a `CONSULTAS_AGENDADAS`.

In [ ]:
print("CONSULTAS AGENDADAS:", CONSULTAS_AGENDADAS)
print()
print("AGENDA Dr. Carlos Mendes:", AGENDA["Dr. Carlos Mendes"])  # 14:00 deve ter sido removido
print("AGENDA Dra. Julia Costa: ", AGENDA["Dra. Julia Costa"])   # 10:00 deve ter sido removido

## 8. O Papel da Docstring

A docstring de cada ferramenta é instrução para o agente. Veja a diferença:

In [ ]:
# Versão ruim — o agente não sabe quando nem como usar
@tool
def verificar_horarios_ruim(nome_medico: str) -> str:
    """Retorna horários."""
    ...

# Versão boa — o agente sabe quando usar e como passar o argumento
@tool
def verificar_horarios_boa(nome_medico: str) -> str:
    """
    Retorna os horários disponíveis de um médico específico.
    Use esta ferramenta antes de agendar, para verificar
    se há horários livres.

    Args:
        nome_medico: Nome completo do médico conforme listado
                     em listar_especialidades().
    """
    ...

# A referência a listar_especialidades() no campo Args ensina o agente
# a encadear as ferramentas na ordem certa:
# 1. listar_especialidades()
# 2. verificar_horarios()
# 3. agendar_consulta()
print("Docstrings vagas causam chamadas erradas ou na ordem errada.")

## Recursos Adicionais

- [Documentação oficial do smolagents — ToolCallingAgent](https://huggingface.co/docs/smolagents/reference/agents#smolagents.ToolCallingAgent)
- [Documentação do decorator @tool](https://huggingface.co/docs/smolagents/reference/tools#smolagents.tool)